In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from libpysal.weights import lat2W
from esda import Moran
np.random.seed(12345)

In [6]:
def extract_year_slice(ds, varname, year):
    da = ds[varname].sel(time=f"{year}-01-01")
    return da.values

In [7]:
def compute_moran(arr2d):
    H, W = arr2d.shape
    
    # NaN 填零（libpysal 需要）
    flat = np.nan_to_num(arr2d.flatten(), nan=0.0)
    
    w = lat2W(H, W)
    w.transform = "r"
    
    mi = Moran(flat, w, permutations=0)
    return mi.I


In [4]:
def compute_all_variables(ds, years,world):
    varnames = list(ds.data_vars)  # 自动遍历所有变量
    results = []

    for year in years:
        print(f"Processing year {year} ...")
        row = {"year": year}

        for varname in varnames:
            print(f"started {varname}")
            arr2d = extract_year_slice(ds, varname, year)

            row[varname] = compute_moran(arr2d)

        results.append(row)

    df = pd.DataFrame(results)
    

    return df


In [ ]:
years = list(range(2010, 2101, 10))
# years.insert(0, 2005)

name = ["agri","forest","grassland"]
mode = ["basin","region"]


for m in mode:

    print(f"calculating {m}_agri_17regions.nc")
    ds = xr.open_dataset(f"../../../NC/{m}_agri_17regions.nc")

    df = compute_all_variables(ds, years, world)



calculating basin_agri_17regions.nc
Processing year 2010 ...
started BRA_basin_agri
started CAN_basin_agri
started CHN_basin_agri
started CIS_basin_agri
started IND_basin_agri
started JPN_basin_agri
started TUR_basin_agri
started USA_basin_agri
started XAF_basin_agri
started XE25_basin_agri
started XER_basin_agri
started XLM_basin_agri
started XME_basin_agri
started XNF_basin_agri
started XOC_basin_agri
started XSA_basin_agri
started XSE_basin_agri
Processing year 2020 ...
started BRA_basin_agri
started CAN_basin_agri
started CHN_basin_agri
started CIS_basin_agri
started IND_basin_agri
started JPN_basin_agri
started TUR_basin_agri
started USA_basin_agri
started XAF_basin_agri
started XE25_basin_agri
started XER_basin_agri
started XLM_basin_agri
started XME_basin_agri
started XNF_basin_agri
started XOC_basin_agri
started XSA_basin_agri
started XSE_basin_agri
Processing year 2030 ...
started BRA_basin_agri
started CAN_basin_agri
started CHN_basin_agri
started CIS_basin_agri
started IND_b

In [17]:
output_csv=f"../../../CSV/moran/{m}_agri_moran_interannual_diff.csv"
df.to_csv(output_csv, index=False)
print(f"Saved CSV to {output_csv}")

Saved CSV to ../../../CSV/moran/region_agri_moran_interannual_diff.csv


In [8]:
# Compute for world
years = list(range(2010, 2101, 10))
# years.insert(0, 2005)

name = ["agri","forest","grassland"]
mode = ["basin","region"]

world = xr.open_dataset("../../../NC/compare_diff.nc")
ds_world = world
df_world = compute_all_variables(ds_world, years, world)

Processing year 2010 ...
started basin_agri_diff
started region_agri_diff
Processing year 2020 ...
started basin_agri_diff
started region_agri_diff
Processing year 2030 ...
started basin_agri_diff
started region_agri_diff
Processing year 2040 ...
started basin_agri_diff
started region_agri_diff
Processing year 2050 ...
started basin_agri_diff
started region_agri_diff
Processing year 2060 ...
started basin_agri_diff
started region_agri_diff
Processing year 2070 ...
started basin_agri_diff
started region_agri_diff
Processing year 2080 ...
started basin_agri_diff
started region_agri_diff
Processing year 2090 ...
started basin_agri_diff
started region_agri_diff
Processing year 2100 ...
started basin_agri_diff
started region_agri_diff


In [9]:
output_csv_world = "../../../CSV/moran/world_moran_interannual_diff.csv"
df_world.to_csv(output_csv_world, index=False)
print(f"Saved CSV to {output_csv_world}")

Saved CSV to ../../../CSV/moran/world_moran_interannual_diff.csv
